In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.55.4 accelerate sentencepiece

Found existing installation: transformers 4.55.4
Uninstalling transformers-4.55.4:
  Successfully uninstalled transformers-4.55.4
  Using cached transformers-4.55.4-py3-none-any.whl.metadata (41 kB)
Using cached transformers-4.55.4-py3-none-any.whl (11.3 MB)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from transformers import pipeline

In [ ]:
import transformers
print(transformers.__version__)

4.55.4


In [ ]:
from transformers.pipelines import SUPPORTED_TASKS

print("summarization" in SUPPORTED_TASKS)
print(list(SUPPORTED_TASKS.keys())[:20])

True
['audio-classification', 'automatic-speech-recognition', 'text-to-audio', 'feature-extraction', 'text-classification', 'token-classification', 'question-answering', 'table-question-answering', 'visual-question-answering', 'document-question-answering', 'fill-mask', 'summarization', 'translation', 'text2text-generation', 'text-generation', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-audio-classification', 'image-classification', 'image-feature-extraction']


In [ ]:
print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
data = {
    'text': [
        "Breaking News: Scientists discover cure for all cancers, available next week!",
        "Local council approves new park development after community consultation.",
        "Shocking! Eating chocolate makes you live forever, say anonymous sources.",
        "Government announces new policy to boost renewable energy sector.",
        "Aliens land in New York, demand to speak to the President. Eyewitnesses report green skin.",
        "Company X reports record profits in Q3, exceeding analyst expectations.",
        "World to end tomorrow, according to ancient prophecy rediscovered by amateur historian.",
        "New study links daily exercise to improved mental health.",
        "Miracle diet pill guarantees 50kg weight loss in a day, no effort required.",
        "Tech giant unveils revolutionary AI chip, promising unprecedented performance."
    ],
    'label': [
        'fake', 'real', 'fake', 'real', 'fake',
        'real', 'fake', 'real', 'fake', 'real'
    ]
}

In [ ]:
df = pd.DataFrame(data)

In [ ]:
# Map labels to numerical values
df['label_encoded'] = df['label'].apply(lambda x: 1 if x == 'fake' else 0)

In [ ]:
X = df['text']
y = df['label_encoded']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# TF-IDF Vectorization
tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_df=0.7)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [ ]:
print("Data prepared and vectorized.")

Data prepared and vectorized.


In [ ]:
# --- 3. ML Model (Fake News Classifier): Train a Logistic Regression model
classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
y_pred = classifier.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)

In [ ]:
print(f"\nML Model Accuracy: {accuracy:.2f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))


ML Model Accuracy: 1.00
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



In [ ]:
print("ML model trained and evaluated.")

ML model trained and evaluated.


In [ ]:
print("Loading GenAI models...")
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")

Loading GenAI models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


In [ ]:
def explain_fake_news(article_text):
    explanation = ""
    # Simple heuristic for fake news characteristics
    if "breaking news" in article_text.lower() or "shocking" in article_text.lower():
        explanation += "The article uses sensational language like 'breaking news' or 'shocking', which is often a characteristic of fake news designed to grab attention. "
    if "anonymous sources" in article_text.lower() or "eyewitnesses report" in article_text.lower():
        explanation += "It relies on vague or anonymous sources ('anonymous sources', 'eyewitnesses report') without verifiable attribution, a common tactic in misinformation. "
    if "cure for all" in article_text.lower() or "live forever" in article_text.lower() or "miracle diet" in article_text.lower() or "world to end" in article_text.lower():
        explanation += "The claims made ('cure for all cancers', 'live forever', 'miracle diet pill', 'world to end') are highly improbable or sensational, lacking scientific or factual basis. "
    if not explanation:
        explanation = "The article contains elements that are often found in misinformation, such as sensational claims or lack of credible sourcing. Further verification is recommended."
    return explanation.strip()

In [ ]:
def summarize_article(article_text):
    # Summarize the article using the pre-trained model
    summary = summarizer(article_text, max_length=100, min_length=30, do_sample=False)
    return summary[0]['summary_text']


In [ ]:
print("GenAI models loaded and functions defined.")

GenAI models loaded and functions defined.


In [ ]:
def detect_and_explain_fake_news(article):
    # Classify the article
    article_tfidf = tfidf_vectorizer.transform([article])
    prediction = classifier.predict(article_tfidf)[0]
    prediction_label = 'fake' if prediction == 1 else 'real'
    print(f"\n--- Analyzing Article ---")
    print(f"Article: {article}")
    print(f"Classification: {prediction_label.upper()}")
    # Summarize the article
    summary = summarize_article(article)
    print(f"Summary: {summary}")

    # Explain if fake
    if prediction_label == 'fake':
        explanation = explain_fake_news(article)
        print(f"Reasoning (why it seems fake): {explanation}")
    else:
        print("Reasoning: The article appears to be real based on our analysis.")

    return {
        'classification': prediction_label,
        'summary': summary,
        'explanation': explanation if prediction_label == 'fake' else "The article appears to be real based on our analysis."
    }

In [ ]:
print("\n--- Running Examples ---")



--- Running Examples ---


In [ ]:
# Example 1: Another Fake News
article1 = "Scientists confirm: Eating pizza every day leads to eternal youth!"
detect_and_explain_fake_news(article1)

Your max_length is set to 100, but your input_length is only 14. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=7)



--- Analyzing Article ---
Article: Scientists confirm: Eating pizza every day leads to eternal youth!
Classification: FAKE
Summary:  Scientists confirm: Eating pizza every day leads to eternal youth . Scientists confirm that eating pizza a day is good for your health . Eating pizza at a pizza shop leads to a healthier lifestyle, scientists say .
Reasoning (why it seems fake): The article contains elements that are often found in misinformation, such as sensational claims or lack of credible sourcing. Further verification is recommended.


{'classification': 'fake',
 'summary': ' Scientists confirm: Eating pizza every day leads to eternal youth . Scientists confirm that eating pizza a day is good for your health . Eating pizza at a pizza shop leads to a healthier lifestyle, scientists say .',
 'explanation': 'The article contains elements that are often found in misinformation, such as sensational claims or lack of credible sourcing. Further verification is recommended.'}

In [ ]:
# Example 2: Another Real News
article2 = "Local government approves budget for new public library construction."
detect_and_explain_fake_news(article2)

Your max_length is set to 100, but your input_length is only 12. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=6)



--- Analyzing Article ---
Article: Local government approves budget for new public library construction.
Classification: REAL
Summary:  Local government approves budget for new public library construction . Local government OKs budget to build public library . Public library will be expanded to accommodate more than 1,000 people .
Reasoning: The article appears to be real based on our analysis.


{'classification': 'real',
 'summary': ' Local government approves budget for new public library construction . Local government OKs budget to build public library . Public library will be expanded to accommodate more than 1,000 people .',
 'explanation': 'The article appears to be real based on our analysis.'}

In [ ]:
# Example 4: Another Fake News
article3 = "Exclusive: Secret alien base discovered under Eiffel Tower, world leaders in panic!"
detect_and_explain_fake_news(article3)

Your max_length is set to 100, but your input_length is only 19. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=9)



--- Analyzing Article ---
Article: Exclusive: Secret alien base discovered under Eiffel Tower, world leaders in panic!
Classification: FAKE
Summary:  Secret alien base discovered under Eiffel Tower, world leaders in panic! Exclusive: Secret alien bases discovered under the French capital's Eiffle Tower . World leaders are in panic over discovery .
Reasoning (why it seems fake): The article contains elements that are often found in misinformation, such as sensational claims or lack of credible sourcing. Further verification is recommended.


{'classification': 'fake',
 'summary': " Secret alien base discovered under Eiffel Tower, world leaders in panic! Exclusive: Secret alien bases discovered under the French capital's Eiffle Tower . World leaders are in panic over discovery .",
 'explanation': 'The article contains elements that are often found in misinformation, such as sensational claims or lack of credible sourcing. Further verification is recommended.'}

In [ ]:
# Example 4: Another Real News
article4 = "Tech company announces major security update for its flagship operating system."
detect_and_explain_fake_news(article4)

Your max_length is set to 100, but your input_length is only 14. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=7)



--- Analyzing Article ---
Article: Tech company announces major security update for its flagship operating system.
Classification: REAL
Summary:  Tech company announces major security update for its flagship operating system . Tech company announced major update to its flagship OS update . The update is the latest in a long line of software updates to the operating system's software .
Reasoning: The article appears to be real based on our analysis.


{'classification': 'real',
 'summary': " Tech company announces major security update for its flagship operating system . Tech company announced major update to its flagship OS update . The update is the latest in a long line of software updates to the operating system's software .",
 'explanation': 'The article appears to be real based on our analysis.'}

After updating the `transformers` library, you should restart the runtime (Runtime > Restart runtime) and then rerun all the cells, especially the one that caused the error (`lcCdUiSbXfWc`), to ensure the changes take effect.